# Project 8: Long-Term Memory Agent

Compare recent-window, episodic, and hybrid semantic/temporal memory. Test
corrections, conflicts, evidence, consolidation, and deletion across implemented stores.

Upgrade v1 prevents deleted-event replay, versions answer caches, and separates
retrieval recall from cited-evidence recall. Context size is measured in words.
An optional validated natural-language extractor has event-bound consent and provenance checks.
Model extraction accuracy and provider-level deletion are not established by fixtures.

In [ ]:
from pathlib import Path
import os, runpy, sys
if sys.prefix == sys.base_prefix:
    raise RuntimeError("Use the isolated notebook kernel from SECURITY_UPGRADE.md; shared Colab packages are not modified")
candidates = [Path.cwd(), Path.cwd()/"project8", Path("/content/ai_agentic_attemptings/project8")]
PROJECT_ROOT = next((p.resolve() for p in candidates if (p/"src/project8_agent").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open this notebook from the cloned project directory")
check = runpy.run_path(str(PROJECT_ROOT.parent/"tools/notebook_environment.py"))["validate"]
check(PROJECT_ROOT.parent/"requirements-notebook-cpu.hashes.txt")
os.chdir(PROJECT_ROOT)
source_root = PROJECT_ROOT/"src"
if str(source_root) not in sys.path:
    sys.path.insert(0, str(source_root))
print("Isolated Project 8 CPU/API environment verified; GPU mode requires a separately approved GPU lock")

In [2]:
import getpass,os,sys
from project8_agent.config import load_config
EVAL_BACKEND="local_gpu"  # deterministic | openai | local_gpu
RUN_FULL_EVAL=True; RUN_API_EVAL=EVAL_BACKEND=="openai"; RUN_LOCAL_GPU_EVAL=EVAL_BACKEND=="local_gpu"; config=load_config(PROJECT_ROOT/"config/default.json")
if RUN_API_EVAL and not os.getenv("OPENAI_API_KEY"):
    if "google.colab" in sys.modules:
        from google.colab import userdata
        key=userdata.get("OPENAI_API_KEY")
    else: key=getpass.getpass("OPENAI_API_KEY (hidden): ")
    if not key: raise RuntimeError("OPENAI_API_KEY required for API mode")
    os.environ["OPENAI_API_KEY"]=key
if RUN_LOCAL_GPU_EVAL:
    import gc,torch
    if not torch.cuda.is_available(): raise RuntimeError("Select a Colab GPU runtime for local_gpu mode")
    stale_names=("source_planner","schema_retriever","local_backend","planners","planner","embedding_retriever","local_answerer")
    released=[name for name in stale_names if globals().pop(name,None) is not None]
    gc.collect(); torch.cuda.empty_cache(); free_bytes,total_bytes=torch.cuda.mem_get_info()
    print({"released_gpu_objects":released,"free_gpu_gib":round(free_bytes/2**30,2),"total_gpu_gib":round(total_bytes/2**30,2)})
print(config.model_dump())

{'released_gpu_objects': [], 'free_gpu_gib': 39.08, 'total_gpu_gib': 39.49}
{'project_id': 'project8', 'seed': 20260802, 'model': 'gpt-5.6-luna', 'reasoning_effort': 'low', 'working_window_size': 6, 'episodic_top_k': 5, 'qa_per_conversation': 40, 'selected_conversations': 2, 'max_model_calls': 300, 'max_output_tokens': 500, 'max_retries': 2, 'max_estimated_cost_usd': 8.0, 'input_price_per_million_usd': 1.0, 'output_price_per_million_usd': 6.0, 'local_model': 'Qwen/Qwen2.5-7B-Instruct', 'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2', 'local_device': 'cuda', 'local_max_new_tokens': 300}


In [3]:
import subprocess,sys
subprocess.run([sys.executable,"tools/fetch_locomo.py"],check=True)
subset_path=PROJECT_ROOT/"data/cache/locomo_subset.json"
print("LoCoMo subset:",subset_path.relative_to(PROJECT_ROOT).as_posix())

LoCoMo subset: data/cache/locomo_subset.json


In [4]:
import json
subset=json.loads(subset_path.read_text()); lifecycle=json.loads((PROJECT_ROOT/"data/lifecycle_cases.json").read_text())
assert len(subset)==2 and sum(len(item["qa"]) for item in subset)==80
print({"conversations":2,"qa":80,"lifecycle_events":len(lifecycle["events"])})

{'conversations': 2, 'qa': 80, 'lifecycle_events': 7}


In [5]:
from project8_agent.memory import MemoryStore
from project8_agent.schemas import MemoryEvent,MemoryQuery
runtime=PROJECT_ROOT/"output/runtime"; runtime.mkdir(parents=True,exist_ok=True); store=MemoryStore(runtime/"memory.sqlite"); store.reset()
for item in lifecycle["events"]: store.ingest(MemoryEvent.model_validate(item))
query=MemoryQuery.model_validate(lifecycle["queries"][0]); print(store.answer(query,"window").model_dump()); print(store.answer(query,"episodic").model_dump())
embedding_retriever=None
if RUN_LOCAL_GPU_EVAL:
    from project8_agent.local_models import EmbeddingEventRetriever
    from project8_agent.locomo import conversation_events
    embedding_retriever=EmbeddingEventRetriever(config.embedding_model,config.local_device)
    sample_events=conversation_events(subset[0]); print({"dense_retrieval":[item["event_id"] for item in embedding_retriever.retrieve(sample_events,subset[0]["qa"][0]["question"],"hybrid",config.working_window_size,config.episodic_top_k)]})

{'query_id': 'Q01', 'system': 'window', 'answer': None, 'evidence_ids': [], 'abstained': True, 'conflict': False, 'context_tokens': 12, 'latency_ms': 0.3891180003847694}
{'query_id': 'Q01', 'system': 'episodic', 'answer': 'Boston', 'evidence_ids': ['E01'], 'abstained': False, 'conflict': False, 'context_tokens': 2, 'latency_ms': 0.6362590011121938}


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

{'dense_retrieval': ['D9:11', 'D14:34', 'D1:3', 'D5:2', 'D8:22']}


In [ ]:
from project8_agent.answer_cache import forget_memory
for event_id in lifecycle["delete_event_ids"]:
    forget_memory(store, event_id, "lifecycle", retrievers=[embedding_retriever] if embedding_retriever else [])
    old = next((item for item in lifecycle["events"] if item["event_id"] == event_id), None)
    if old: assert not store.ingest(MemoryEvent.model_validate(old))
correction=MemoryQuery.model_validate(lifecycle["queries"][1]); conflict=MemoryQuery.model_validate(lifecycle["queries"][2])
print(store.answer(correction,"hybrid").model_dump()); print(store.answer(conflict,"hybrid").model_dump()); assert all(store.deletion_verified(e) for e in lifecycle["delete_event_ids"])

# Authored extraction fixture: validates storage controls, not LLM extraction quality.
from project8_agent.extraction import Extraction,ExtractedFact,RetentionConsent,event_digest,ingest_extraction
natural_store=MemoryStore(runtime/"natural_demo.sqlite")
event=MemoryEvent(event_id="natural-demo-1",session_id="demo",timestamp="2026-01-01T12:00:00+00:00",speaker="demo-user",text="I live in Boston.")
consent=RetentionConsent(subject=event.speaker,event_digest=event_digest(event),allowed_keys=["home_city"],retain=True)
extraction=Extraction(facts=[ExtractedFact(key="home_city",value="Boston",source_event_id=event.event_id,evidence_quote=event.text)])
ingest_extraction(natural_store,event,extraction,consent,"authored-fixture-v1")
forget_memory(natural_store,event.event_id,"natural-demo")
assert natural_store.deletion_verified(event.event_id)
assert not ingest_extraction(natural_store,event,extraction,consent,"authored-fixture-v1")
print("Natural-event consent/provenance/deletion fixture passed; no provider call was made")

In [7]:
import os,subprocess,sys
test_env=os.environ.copy(); test_env["PYTEST_DISABLE_PLUGIN_AUTOLOAD"]="1"
try:
    result=subprocess.run([sys.executable,"-m","pytest","-q","tests"],text=True,capture_output=True,env=test_env,timeout=120)
except subprocess.TimeoutExpired as exc:
    raise RuntimeError("Project 8 tests exceeded the 120-second Colab limit") from exc
print(result.stdout)
if result.returncode: print(result.stderr); raise RuntimeError("Project 8 tests failed")

............                                                             [100%]
12 passed in 0.53s



In [ ]:
from runpy import run_path
RunManifest = run_path(str(PROJECT_ROOT.parent / 'tools/run_manifest.py'))['RunManifest']
from project8_agent.evaluation import evaluate_lifecycle
from project8_agent.locomo import CachedLocalAnswerer, CachedOpenAIAnswerer, evaluate_locomo
from datetime import datetime, timezone
if not RUN_FULL_EVAL: print("Set RUN_FULL_EVAL=True after P08-C07 passes.")
else:
    result_dir=PROJECT_ROOT/"output/runs/upgrade-v2"/EVAL_BACKEND/datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    with RunManifest(PROJECT_ROOT, result_dir, config.model_dump()) as run_manifest:
        lifecycle_summary=evaluate_lifecycle(PROJECT_ROOT/"data/lifecycle_cases.json",runtime/"evaluation.sqlite",runtime/"lifecycle_output")
        if RUN_LOCAL_GPU_EVAL:
            import copy,random
            # Two retrievers, three memory systems: bound worst-case calls before generation.
            qa_limit=min(80,config.max_model_calls//6)
            if qa_limit < len(subset): raise RuntimeError("Model-call budget cannot cover every conversation")
            rng=random.Random(2026); ablation_subset=copy.deepcopy(subset)
            for i,sample in enumerate(ablation_subset):
                count=qa_limit//len(subset)+(i < qa_limit%len(subset))
                sample["qa"]=rng.sample(sample["qa"],min(count,len(sample["qa"])))
            ablation_path=result_dir/"ablation_subset.json"
            ablation_path.write_text(json.dumps(ablation_subset,indent=2))
            local_answerer=CachedLocalAnswerer(config,runtime/"upgrade_v1_local_answer_cache.json")
            summaries={}
            for retrieval_name, retriever in [("lexical", None), ("dense", embedding_retriever)]:
                summaries[retrieval_name]=evaluate_locomo(ablation_path,config,runtime/"unused.json",result_dir/retrieval_name,lifecycle_summary,answerer=local_answerer,retriever=retriever,evaluation_mode="locomo_local_gpu")
            print("Same-answerer retrieval ablation:", summaries)
            result_dir=result_dir/"dense"; summary=summaries["dense"]
        elif RUN_API_EVAL:
            summary=evaluate_locomo(subset_path,config,runtime/"upgrade_v1_api_answer_cache.json",result_dir,lifecycle_summary)
        else:
            summary=evaluate_lifecycle(PROJECT_ROOT/"data/lifecycle_cases.json",runtime/"evaluation.sqlite",result_dir)
        print(summary)
        run_manifest.record(summary, {name: getattr(globals().get(name), 'model_revision', None) for name in ('source_planner', 'local_backend', 'embedding_retriever')})

In [ ]:
import json
if "result_dir" not in globals(): raise RuntimeError("Run P08-C08 first; historical results are not reused")
path=result_dir/"project8_representative_samples.json"; print(json.loads(path.read_text()) if path.exists() else "Run P08-C08 first.")

In [ ]:
import subprocess,sys
if "result_dir" not in globals(): raise RuntimeError("Run P08-C08 first; historical results are not reused")
summary_path=result_dir/"project8_final_summary.json"
if summary_path.exists():
    subprocess.run([sys.executable,"tools/generate_report.py","--summary",str(summary_path),"--output",str(result_dir/"project8_report.pdf")],check=True); subprocess.run([sys.executable,"tools/validate_project.py"]+([] if RUN_LOCAL_GPU_EVAL else ["--require-results"]),check=True)
else: print("Measured summary absent; report generation skipped.")

## LongMemEval-S: Temporal Reasoning, Updates And Abstention

The prepared cleaned-S slice is checksum-verified and stratified before model
evaluation. Oracle flags, answer-session IDs and answer-revealing ID prefixes
never enter retrieval/generation. All arms use the same answerer and context-token
cap. Lexical F1 is only a proxy; official semantic grading is separate and not run.
See `BENCHMARKS.md`. Model execution below remains disabled by default.

In [ ]:
RUN_LONGMEMEVAL = os.environ.get("PORTFOLIO_LONGMEMEVAL") == "1"
if RUN_LONGMEMEVAL:
    import json
    from datetime import datetime, timezone
    from runpy import run_path
    from project8_agent.longmemeval import evaluate as evaluate_longmemeval
    from project8_agent.locomo import CachedLocalAnswerer, retrieve
    if not RUN_LOCAL_GPU_EVAL:
        raise RuntimeError("This cell needs the explicitly selected local answerer and its actual tokenizer")
    longmem_rows = json.loads((PROJECT_ROOT / "data/cache/longmemeval-v2/selected.json").read_text())
    longmem_run = PROJECT_ROOT / "output/runs/upgrade-v2/longmemeval" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    RunManifest = run_path(str(PROJECT_ROOT.parent / "tools/run_manifest.py"))["RunManifest"]
    with RunManifest(PROJECT_ROOT, longmem_run, config.model_dump()) as provenance:
        answerer = CachedLocalAnswerer(config, longmem_run / "runtime/answer_cache.json")
        tokenizer = answerer.backend.tokenizer
        count_context_tokens = lambda context: len(tokenizer.encode(json.dumps(context), add_special_tokens=False))
        retrievers = {mode: (lambda events, question, mode=mode: retrieve(events, question, mode, top_k=12))
                      for mode in ["window", "episodic", "hybrid"]}
        longmem_summary = evaluate_longmemeval(longmem_rows, answerer, retrievers, count_context_tokens,
            config.local_model + "@" + answerer.backend.model_revision, longmem_run / "results")
        provenance.record(longmem_summary, {"model_revision": answerer.backend.model_revision})
    print(longmem_summary)
else:
    print("LongMemEval model evaluation is deferred; preparation and parser tests need no GPU or API key.")

In [44]:
PROJECT = 8
BUNDLE_SHA256 = '9b771d0c1fe5f354debc4e318c1f05590d60f7cc4f91e8b5afd228cefd57ee5f'
BUNDLE = '...'
from datetime import datetime, timezone
import base64
import gzip
import hashlib
import json
import os
from pathlib import Path
import subprocess
import sys

from google.colab import drive

mount = Path("/content/drive")
if not mount.is_mount():
    drive.mount(str(mount), timeout_ms=180000)
if not mount.is_mount():
    raise RuntimeError("Connect this notebook to the mounted MyPro runtime")
gpu = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True).strip()
if "A100" not in gpu:
    raise RuntimeError(f"Expected the A100, found {gpu}")
if globals().get("inference_worker") is not None and inference_worker.poll() is None:
    raise RuntimeError("This notebook already has an active evaluation worker")
if subprocess.check_output(
    ["nvidia-smi", "--query-compute-apps=pid", "--format=csv,noheader"], text=True).strip():
    raise RuntimeError("Another process is using the GPU")

runtime = Path("/content/portfolio_inference_v2")
bundle_dir = runtime / f"project{PROJECT}" / BUNDLE_SHA256
tools_dir = bundle_dir / "tools"
tools_dir.mkdir(parents=True, exist_ok=True)
bundle = gzip.decompress(base64.b64decode(BUNDLE))
if hashlib.sha256(bundle).hexdigest() != BUNDLE_SHA256:
    raise RuntimeError("Notebook package hash mismatch")
for name, contents in json.loads(bundle).items():
    destination = (tools_dir / name).resolve()
    if not destination.is_relative_to(tools_dir):
        raise ValueError("Unexpected path in notebook package")
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(contents)

bootstrap = runtime / "bootstrap"
subprocess.run([sys.executable, "-m", "pip", "install", "--target", str(bootstrap),
                "--no-deps", "uv==0.12.17"], check=True, timeout=180)
uv_cache = Path("/content/project1_training_v2/uv-cache")
if not uv_cache.is_dir():
    uv_cache = runtime / "uv-cache"
uv_env = dict(os.environ, PYTHONPATH=str(bootstrap), UV_CACHE_DIR=str(uv_cache),
              UV_HTTP_RETRIES="1")

def uv(*arguments):
    subprocess.run([sys.executable, "-m", "uv", *map(str, arguments)], env=uv_env,
                   check=True, timeout=600)

venv = runtime / "venv"
python = venv / "bin/python"
lock = tools_dir / "requirements-inference-v2.hashes.txt"
fingerprint = hashlib.sha256(lock.read_bytes()).hexdigest()
marker = venv / ".portfolio-lock"
if not python.is_file():
    uv("venv", "--python", "3.11.17", venv)
if not marker.is_file() or marker.read_text() != fingerprint:
    uv("pip", "install", "--python", python, "--require-hashes", "-r", lock)
    uv("pip", "check", "--python", python)
    marker.write_text(fingerprint)

project_root = mount / f"MyDrive/projects/MyProject{PROJECT}"
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
run_parent = project_root / "outputs/runs/upgrade-v2"
run_parent.mkdir(parents=True, exist_ok=True)
run_dir = run_parent / run_id
inference_log = run_parent / f"{run_id}.log"
inference_env = {
    k: v for k, v in os.environ.items()
    if not any(word in k.upper() for word in ("TOKEN", "SECRET", "PASSWORD", "API_KEY"))
}
inference_env.update(
    HF_HUB_DISABLE_IMPLICIT_TOKEN="1", HF_HUB_DISABLE_XET="1",
    HF_HUB_DISABLE_TELEMETRY="1", PYTHONUNBUFFERED="1", OTEL_SDK_DISABLED="true",
)
with inference_log.open("x") as log:
    inference_worker = subprocess.Popen(
        [str(python), str(tools_dir / "run_inference_cells_v2.py"), "--project",
         str(PROJECT), "--cells", str(tools_dir / "cells.json"), "--output", str(run_dir)],
        env=inference_env, stdout=log, stderr=subprocess.STDOUT, start_new_session=True,
    )
print({"project": PROJECT, "run_id": run_id, "pid": inference_worker.pid})


{'project': 8, 'run_id': '20261118T102213022610Z', 'pid': 119091}


In [46]:
import json
import subprocess

manifest_path = run_dir / "run_manifest.json"
if manifest_path.is_file():
    print(json.dumps(json.loads(manifest_path.read_text()), indent=2))
exit_code = inference_worker.poll()
print("Worker exit:", exit_code)
if exit_code is None:
    gpu = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=utilization.gpu,memory.used", "--format=csv,noheader"], text=True)
    print("GPU utilization, memory in use:", gpu.strip())
elif exit_code != 0:
    with inference_log.open("rb") as log:
        log.seek(max(0, inference_log.stat().st_size - 4096))
        print(log.read().decode(errors="replace"))
elif PROJECT == 5:
    keys = ("value_result_accuracy_pct", "value_result_cases", "pii_leak_rate_pct",
            "planning_error_pct", "execution_or_repair_error_pct", "end_to_end_p50_ms")
    for arm in ("baseline", "candidate"):
        summary = json.loads((run_dir / arm / "project5_final_summary.json").read_text())
        metrics = {system: {key: summary[system][key] for key in keys}
                   for system in ("baseline", "governed")}
        payload = {"model": summary["config"]["local_model"], "cases": summary["test_questions"],
                   "usage": summary["planner_usage"], "metrics": metrics}
        print(f"SQL model {arm}:", json.dumps(payload))
elif PROJECT == 8:
    summary = json.loads((run_dir / "longmemeval/results/summary.json").read_text())
    print("LongMemEval:", json.dumps(summary))


{
  "project": 8,
  "run_id": "20261118T102213022610Z",
  "status": "complete",
  "gpu": "NVIDIA A100-SXM4-40GB",
  "python": "3.11.17",
  "lock_sha256": "afc112d1c55172cee298cc26e4e3119e2e9d8506dda21494050212c69ed3c906",
  "cell_bundle_sha256": "450c9ebc3767f4cad66eb0460fd4a61109ea8443e368d0a26817b72a78844232",
  "completed_cells": [
    3,
    13
  ],
  "active_cell": 13,
  "seconds": 122.68513515900122,
  "peak_gpu_allocated_bytes": 15662598144
}
Worker exit: 0
LongMemEval: {"version": "longmemeval-s-v1", "dataset": "xiaowu0162/longmemeval-cleaned", "revision": "98d7416c24c778c2fee6e6f3006e7a073259d48f", "seed": 2026, "question_ids": ["6e984302", "eac54add", "dfde3500", "e493bb7c", "e5ba910e_abs", "gpt4_93159ced_abs"], "tokenizer": "Qwen/Qwen2.5-7B-Instruct@a09a35458c702b33eeacc393d103063234e8bc28", "context_token_budget": 2048, "summary": {"window": {"temporal-reasoning": {"count": 2, "errors": 0, "token_f1_proxy": 0.0, "abstention_correct": null, "session_recall": 0.0, "retrieved_